# Seq vs OMP vs FF — 3×3 Composite Plots (Manual Pairs)

This notebook:
1) Loads non-MPI CSVs, computes `total_ms`, **prints sample rows where `sorted != 1`**, then filters to `sorted == 1`.
2) Builds **3×3 composite images** for **Total time**, **Speedup**, and **Efficiency** over manual `(records, payload_max)` pairs:
   - `RECORDS = [1_000_000, 10_000_000, 100_000_000]`
   - `PAYLOAD_MAX = [8, 32, 128]`


In [13]:

# 1) Load CSVs, compute total time, show unsorted sample, keep only sorted==1
from pathlib import Path
import glob
import pandas as pd
import numpy as np
from itertools import product

# ---- CONFIG ----
CSV_GLOB = "../results/*.csv"   # adjust to your layout
CSV_GLOB_FALLBACK = "*.csv"     # fallback if needed
OUT_DIR = Path("plots_seq_omp_ff_manual")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Manual pairs
RECORDS = [1_000_000, 10_000_000, 100_000_000]
PAYLOAD_MAX = [8, 32, 128]
TOP_PAIRS = list(product(RECORDS, PAYLOAD_MAX))  # 9 pairs, fixed order

def infer_impl_from_name(path: str) -> str:
    name = Path(path).name.lower()
    if "mpi" in name:
        return "mpi"
    if "fastflow" in name or "ff" in name:
        return "ff"
    if "openmp" in name or "omp" in name:
        return "omp"
    if "seq" in name or "sequential" in name:
        return "seq"
    return "unknown"

# Load CSVs
csvs = glob.glob(CSV_GLOB, recursive=True)
if not csvs:
    csvs = glob.glob(CSV_GLOB_FALLBACK, recursive=False)
if not csvs:
    raise SystemExit("No CSV files found. Set CSV_GLOB correctly and re-run.")

frames = []
for p in csvs:
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"Skipping {p}: {e}")
        continue
    if "impl" not in df.columns:
        df["impl"] = infer_impl_from_name(p)
    df["source_csv"] = p
    frames.append(df)

data_raw = pd.concat(frames, ignore_index=True)

# Focus on non-MPI implementations
data = data_raw[data_raw["impl"].isin(["seq","omp","ff"])].copy()

# Ensure required columns exist
required = ["reading_and_sorting_ms","writing_ms","threads","records","payload_max","sorted","impl"]
missing = [c for c in required if c not in data.columns]
if missing:
    raise SystemExit(f"Missing required columns in data: {missing}")

# Coerce numerics
num_cols = ["reading_and_sorting_ms","writing_ms","threads","records","payload_max","sorted"]
for c in num_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

# Compute total time (ms)
data["total_ms"] = data["reading_and_sorting_ms"]# + data["writing_ms"] #!!!!!!!!!!!!!!!!

# --- DEBUG: show unsorted rows BEFORE filtering ---
total_rows = len(data)
not_sorted = data[data["sorted"] != 1].copy()
ns_rows = len(not_sorted)
print(f"Loaded rows (non-MPI): {total_rows} — rows with sorted != 1: {ns_rows}")

if ns_rows == 0:
    print("All rows have sorted == 1.")

# Now filter only correctly sorted runs
data = data[data["sorted"] == 1].copy()
after = len(data)
print(f"Kept sorted==1 rows: {after}")

# Persist globals for later cells
OUT_DIR_STR = str(OUT_DIR)
TOP_PAIRS = TOP_PAIRS


Loaded rows (non-MPI): 216 — rows with sorted != 1: 0
All rows have sorted == 1.
Kept sorted==1 rows: 216


In [14]:

# 2) Composite image for TOTAL TIME (9 panels) over manual TOP_PAIRS
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

OUT_DIR = Path(OUT_DIR_STR)

def aggregate_time(df):
    return (df.groupby(["impl","threads"], as_index=False)
              .agg(total_ms=("total_ms","median")))

def plot_time_panel(df_pair, rec, pay, panel_path):
    plt.figure(figsize=(6,4))
    agg = aggregate_time(df_pair)
    any_curve = False
    for impl in ["seq","omp","ff"]:
        dfi = agg[agg["impl"]==impl].sort_values("threads")
        if dfi.empty:
            continue
        any_curve = True
        plt.plot(dfi["threads"], dfi["total_ms"], marker="o", label=impl.upper())
    plt.xlabel("Threads")
    plt.ylabel("Total time (ms)")
    plt.title(f"N={rec}, payload_max={pay}")
    plt.grid(True)
    if any_curve:
        plt.legend()
    plt.tight_layout()
    plt.savefig(panel_path, bbox_inches="tight")
    plt.close()

panel_paths = []
for idx, (rec, pay) in enumerate(TOP_PAIRS):
    dfp = data[(data["records"]==rec) & (data["payload_max"]==pay)].copy()
    if dfp.empty:
        panel_path = Path(OUT_DIR) / f"time_panel_{idx}_N{int(rec)}_K{int(pay)}_EMPTY.png"
        plt.figure(figsize=(6,4))
        plt.title(f"N={rec}, payload_max={pay}\n(no data)")
        plt.axis('off')
        plt.savefig(panel_path, bbox_inches="tight")
        plt.close()
    else:
        panel_path = Path(OUT_DIR) / f"time_panel_{idx}_N{int(rec)}_K{int(pay)}.png"
        plot_time_panel(dfp, rec, pay, panel_path)
    panel_paths.append(panel_path)

imgs = [Image.open(p) for p in panel_paths]
w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)
norm = []
for im in imgs:
    if im.size != (w,h):
        im = im.resize((w,h))
    norm.append(im)

cols, rows = 3, 3
mosaic = Image.new("RGB", (cols*w, rows*h), "white")
for i, im in enumerate(norm):
    r, c = divmod(i, cols)
    mosaic.paste(im, (c*w, r*h))
out_path = Path(OUT_DIR) / "composite_time_3x3.png"
mosaic.save(out_path)
out_path


PosixPath('plots_seq_omp_ff_manual/composite_time_3x3.png')

In [15]:

# 3) Composite image for SPEEDUP (9 panels) — baseline seq @ threads=1
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

OUT_DIR = Path(OUT_DIR_STR)

def plot_speedup_panel(df_pair, rec, pay, panel_path):
    base = df_pair[(df_pair["impl"]=="seq") & (df_pair["threads"]==1)]
    if base.empty:
        plt.figure(figsize=(6,4))
        plt.title(f"N={rec}, payload_max={pay}\n(no seq@1 baseline)")
        plt.axis('off')
        plt.savefig(panel_path, bbox_inches="tight")
        plt.close()
        return
    Tref = base["total_ms"].median()
    agg = (df_pair.groupby(["impl","threads"], as_index=False)
                 .agg(total_ms=("total_ms","median")))
    agg["speedup"] = Tref / agg["total_ms"]

    plt.figure(figsize=(6,4))
    any_curve = False
    for impl in ["seq","omp","ff"]:
        dfi = agg[agg["impl"]==impl].sort_values("threads")
        if dfi.empty:
            continue
        any_curve = True
        plt.plot(dfi["threads"], dfi["speedup"], marker="o", label=impl.upper())
    xs = sorted(agg["threads"].unique())
    if xs:
        plt.plot(xs, xs, linestyle="--", label="Ideal (S=p)")
    plt.xlabel("Threads")
    plt.ylabel("Speedup (vs seq@1)")
    plt.title(f"N={rec}, payload_max={pay}")
    plt.grid(True)
    if any_curve:
        plt.legend()
    plt.tight_layout()
    plt.savefig(panel_path, bbox_inches="tight")
    plt.close()

panel_paths = []
for idx, (rec, pay) in enumerate(TOP_PAIRS):
    dfp = data[(data["records"]==rec) & (data["payload_max"]==pay)].copy()
    panel_path = Path(OUT_DIR) / f"speed_panel_{idx}_N{int(rec)}_K{int(pay)}.png"
    if dfp.empty:
        plt.figure(figsize=(6,4))
        plt.title(f"N={rec}, payload_max={pay}\n(no data)")
        plt.axis('off')
        plt.savefig(panel_path, bbox_inches="tight")
        plt.close()
    else:
        plot_speedup_panel(dfp, rec, pay, panel_path)
    panel_paths.append(panel_path)

imgs = [Image.open(p) for p in panel_paths]
w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)
norm = []
for im in imgs:
    if im.size != (w,h):
        im = im.resize((w,h))
    norm.append(im)

cols, rows = 3, 3
mosaic = Image.new("RGB", (cols*w, rows*h), "white")
for i, im in enumerate(norm):
    r, c = divmod(i, cols)
    mosaic.paste(im, (c*w, r*h))
out_path = Path(OUT_DIR) / "composite_speedup_3x3.png"
mosaic.save(out_path)
out_path


PosixPath('plots_seq_omp_ff_manual/composite_speedup_3x3.png')

In [16]:

# 4) Composite image for EFFICIENCY (9 panels) — E = S/p
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

OUT_DIR = Path(OUT_DIR_STR)

def plot_efficiency_panel(df_pair, rec, pay, panel_path):
    base = df_pair[(df_pair["impl"]=="seq") & (df_pair["threads"]==1)]
    if base.empty:
        plt.figure(figsize=(6,4))
        plt.title(f"N={rec}, payload_max={pay}\n(no seq@1 baseline)")
        plt.axis('off')
        plt.savefig(panel_path, bbox_inches="tight")
        plt.close()
        return
    Tref = base["total_ms"].median()
    agg = (df_pair.groupby(["impl","threads"], as_index=False)
                 .agg(total_ms=("total_ms","median")))
    agg["speedup"] = Tref / agg["total_ms"]
    agg["efficiency"] = agg["speedup"] / agg["threads"]

    plt.figure(figsize=(6,4))
    any_curve = False
    for impl in ["seq","omp","ff"]:
        dfi = agg[agg["impl"]==impl].sort_values("threads")
        if dfi.empty:
            continue
        any_curve = True
        plt.plot(dfi["threads"], dfi["efficiency"], marker="o", label=impl.upper())
    plt.xlabel("Threads")
    plt.ylabel("Efficiency (S/p)")
    plt.title(f"N={rec}, payload_max={pay}")
    plt.grid(True)
    if any_curve:
        plt.legend()
    plt.tight_layout()
    plt.savefig(panel_path, bbox_inches="tight")
    plt.close()

panel_paths = []
for idx, (rec, pay) in enumerate(TOP_PAIRS):
    dfp = data[(data["records"]==rec) & (data["payload_max"]==pay)].copy()
    panel_path = Path(OUT_DIR) / f"eff_panel_{idx}_N{int(rec)}_K{int(pay)}.png"
    if dfp.empty:
        plt.figure(figsize=(6,4))
        plt.title(f"N={rec}, payload_max={pay}\n(no data)")
        plt.axis('off')
        plt.savefig(panel_path, bbox_inches="tight")
        plt.close()
    else:
        plot_efficiency_panel(dfp, rec, pay, panel_path)
    panel_paths.append(panel_path)

imgs = [Image.open(p) for p in panel_paths]
w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)
norm = []
for im in imgs:
    if im.size != (w,h):
        im = im.resize((w,h))
    norm.append(im)

cols, rows = 3, 3
mosaic = Image.new("RGB", (cols*w, rows*h), "white")
for i, im in enumerate(norm):
    r, c = divmod(i, cols)
    mosaic.paste(im, (c*w, r*h))
out_path = Path(OUT_DIR) / "composite_efficiency_3x3.png"
mosaic.save(out_path)
out_path


PosixPath('plots_seq_omp_ff_manual/composite_efficiency_3x3.png')